# CLINICAL TRIALS RISK ANALYSIS
---
## 02 - Data Preprocessing
This notebook loads raw clinical trial records, identifies data quality issues, performs data cleaning, handles missing values and outliers, and prepares the dataset for subsequent exploratory data analysis and modeling.

The goals of this notebook include:
- Understanding the meaning and structure of each feature  
- Checking and correcting data types  
- Handling missing and inconsistent values  
- Removing outliers in critical numeric variables  
- Validating the cleaned dataset  
- Exporting a clean dataset for further analysis  

### Imports & Config

In [1]:
import sys
import os

# Path to project root (folder that contains "src/")
project_root = os.path.abspath("..")
sys.path.append(project_root)

masked = project_root.replace(os.path.expanduser("~"), "~")
print("Project root added:", masked)

Project root added: ~/Clinical-Trial-Failure-Prediction


In [2]:
import pandas as pd

from src.data.data_cleaner import (
    clean_dataframe,
    validate_values,
    remove_outliers_iqr,
    fill_missing_median
)

RAW_PATH = "../data/raw/clinical_trials_raw.csv"

### Load Raw Data

In [3]:
df = pd.read_csv(RAW_PATH)
df.shape

(100000, 29)

### What is the meaning of each column?
The variables listed below represent the subset of features that has been selected for preprocessing and analysis.  
Each feature captures a specific structural or operational aspect of a clinical trial, and together they form the basis for subsequent risk assessment and predictive modeling.  

A concise description of each selected variable is provided as follows:

| Feature Name                      | Meaning                                                                                |
|-----------------------------------|----------------------------------------------------------------------------------------|
| **nct_id**                        | Unique identifier for each clinical trial on ClinicalTrials.gov.                       |
| **overall_status**                | Current trial status (e.g., Completed, Recruiting, Terminated, Withdrawn).             |
| **start_date**                    | The date when the trial officially started.                                            |
| **primary_completion_date**       | Date when data collection for primary outcomes was completed (anticipated or actual).  |
| **completion_date**               | Date when the entire study was completed.                                              |
| **study_type**                    | Type of study: Interventional or Observational.                                        |
| **phases**                        | Clinical trial phase: Phase 1, Phase 2, Phase 3, or Phase 4.                           |
| **allocation**                    | Method of subject assignment: Randomized or Non-randomized.                            |
| **intervention_model**            | Study model used: Parallel, Crossover, Sequential, etc.                                |
| **masking**                       | Blinding level: Open-label, Single-blind, Double-blind.                                |
| **primary_purpose**               | Main purpose of the study: Treatment, Prevention, Diagnostic, etc.                     |
| **conditions**                    | List of medical conditions or diseases studied in the trial.                           |
| **num_conditions**                | Number of conditions included in the trial.                                            |
| **num_arms**                      | Number of study arms (treatment groups).                                               |
| **num_interventions**             | Number of interventions used in the study.                                             |
| **num_primary_outcomes**          | Count of primary outcomes defined for the trial.                                       |
| **num_secondary_outcomes**        | Count of secondary outcomes defined for the trial.                                     |
| **responsible_party_type**        | Legal responsible party for the study (e.g., Sponsor or Principal Investigator).       |
| **lead_sponsor_class**            | Category of the lead sponsor (Industry, NIH, U.S. Fed, University, etc.).              |
| **num_collaborators**             | Number of collaborating organizations or institutions.                                 |
| **healthy_volunteers**            | Indicates whether healthy volunteers are allowed to participate.                       |
| **sex_eligibility**               | Eligible sex categories: All, Female, or Male.                                         |
| **min_age**                       | Minimum age allowed for participants.                                                  |
| **max_age**                       | Maximum age allowed for participants.                                                  |
| **num_locations**                 | Number of study sites where the trial is conducted.                                    |
| **num_countries**                 | Number of countries in which the trial is conducted.                                   |
| **num_condition_mesh_terms**      | Number of MeSH (Medical Subject Headings) terms describing the conditions.             |
| **num_intervention_mesh_terms**   | Number of MeSH terms describing the interventions.                                     |
| **has_results**                   | Indicates whether study results have been posted on ClinicalTrials.gov.                |


### What is the current data type of each column?
Inspecting data types helps identify:
- incorrect types (e.g., numeric stored as string)
- categorical fields that should be encoded
- date fields that need conversion

In [4]:
df.dtypes

nct_id                         object
overall_status                 object
start_date                     object
completion_date                object
primary_completion_date        object
study_type                     object
phases                         object
allocation                     object
intervention_model             object
masking                        object
primary_purpose                object
conditions                     object
num_conditions                  int64
num_arms                        int64
num_interventions               int64
num_primary_outcomes            int64
num_secondary_outcomes          int64
responsible_party_type         object
lead_sponsor_class             object
num_collaborators               int64
healthy_volunteers             object
sex_eligibility                object
min_age                        object
max_age                        object
num_locations                   int64
num_countries                   int64
num_conditio

### Are there columns having inappropriate data types?
**Columns Already Using Appropriate Data Types**
- **int64**: all numeric count columns  
- **bool**: has_results  

These columns already match the correct semantic data types and require no correction.

**Columns With Inappropriate or Suboptimal Data Types**
| Column Name          | Issue Description |
|----------------------|------------------|
| **start_date, primary_completion_date, completion_date**   | Stored as `object`, should be datetime. | 
| **phases**           | Stored as `object` (string), but represents a structured categorical field (e.g., Phase 1, Phase 2). |
| **healthy_volunteers** | Stored as `object`, but logically a boolean. |
| **sex_eligibility**  | Stored as `object`, should be categorical. |
| **min_age**          | Stored as `object`, values include strings like `"18 Years"`, `"Adult"` $\rightarrow$ should be converted to numeric. |
| **max_age**          | Same issue as `min_age`, contains non-numeric strings. |

**Summary**  

Most numerical and date columns are correctly typed. However, several datetime, categorical and age-related fields are currently stored as `object` and need further preprocessing to ensure accurate modeling. Key columns requiring cleaning include: `start_date`, `primary_completion_date`, `completion_date`, `phases`, `healthy_volunteers`, `sex_eligibility`, `min_age`, and `max_age`.

### Clean Data Types
The `clean_dataframe()` preprocessing routine is applied to correct type inconsistencies.  
This correction includes:
- conversion of date fields from object to datetime64[ns], enabling proper sorting, filtering, and time-based calculations
- conversion of trial phases into categorical and numeric forms  
- transformation of eligibility fields (e.g., sex eligibility, healthy volunteer status, minimum and maximum age) into consistent encodings  
- extraction of numeric values from textual age fields  
- replacement of empty-string categorical values with proper missing values (`NA`) to ensure consistent downstream imputation  

After cleaning, column data types more accurately reflect the characteristics of the underlying variables.

In [5]:
df = clean_dataframe(df)
df.head()

,nct_id,overall_status,start_date,completion_date,primary_completion_date,study_type,phases,allocation,intervention_model,masking,...,healthy_volunteers,sex_eligibility,min_age,max_age,num_locations,num_countries,num_condition_mesh_terms,num_intervention_mesh_terms,has_results,phase_num
0,NCT02979535,COMPLETED,2016-11-16,2019-03-25,2019-03-25,INTERVENTIONAL,PHASE3,RANDOMIZED,PARALLEL,NONE,...,True,FEMALE,9.0,14.0,1,1,2,2,True,3.0
1,NCT02074735,COMPLETED,2014-04-01,2016-11-01,2016-10-01,INTERVENTIONAL,PHASE4,RANDOMIZED,PARALLEL,TRIPLE,...,False,ALL,18.0,75.0,1,1,1,2,True,4.0
2,NCT05242835,RECRUITING,2023-02-03,2027-03-01,2027-01-01,INTERVENTIONAL,NaN,RANDOMIZED,PARALLEL,QUADRUPLE,...,False,ALL,18.0,60.0,1,1,2,0,False,NaN
3,NCT07113535,NOT_YET_RECRUITING,2025-08-01,2027-12-31,2027-06-30,INTERVENTIONAL,NaN,RANDOMIZED,PARALLEL,SINGLE,...,False,ALL,NaN,NaN,1,1,2,0,False,NaN
4,NCT01684735,COMPLETED,2012-03-01,2014-12-01,2014-12-01,OBSERVATIONAL,NaN,NaN,NaN,NaN,...,False,FEMALE,NaN,NaN,1,1,1,0,False,NaN


In [6]:
df.dtypes

nct_id                                 object
overall_status                         object
start_date                     datetime64[ns]
completion_date                datetime64[ns]
primary_completion_date        datetime64[ns]
study_type                             object
phases                               category
allocation                             object
intervention_model                     object
masking                                object
primary_purpose                        object
conditions                             object
num_conditions                          int64
num_arms                                int64
num_interventions                       int64
num_primary_outcomes                    int64
num_secondary_outcomes                  int64
responsible_party_type                 object
lead_sponsor_class                     object
num_collaborators                       int64
healthy_volunteers                    boolean
sex_eligibility                   

## How are values distributed across columns?
The distribution of values is examined to characterize the statistical properties of the dataset.  
This analysis assists in identifying skewness, imbalance, unusual patterns, and potential sources of noise.

### Numeric Summary
Summary statistics (mean, standard deviation, quartiles, and ranges) are generated for all numeric fields.  
These statistics provide insight into the central tendency and variability of the dataset.

In [7]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
start_date,99051,2016-11-16 08:56:50.606657024,1963-01-01 00:00:00,2012-06-01 00:00:00,2018-03-15 00:00:00,2022-04-28 00:00:00,2097-11-01 00:00:00,NaN
completion_date,96959,2020-01-04 17:09:56.775956992,1976-11-01 00:00:00,2015-11-01 00:00:00,2021-03-01 00:00:00,2024-12-08 00:00:00,2100-12-31 00:00:00,NaN
primary_completion_date,96085,2019-10-14 08:02:21.774470656,1979-08-01 00:00:00,2015-09-01 00:00:00,2020-11-30 00:00:00,2024-07-31 00:00:00,2100-12-01 00:00:00,NaN
num_conditions,100000.0,1.78083,0.0,1.0,1.0,2.0,185.0,2.179363
num_arms,100000.0,1.84053,0.0,1.0,2.0,2.0,49.0,1.461992
num_interventions,100000.0,1.70014,0.0,1.0,1.0,2.0,60.0,1.448981
num_primary_outcomes,100000.0,1.95867,0.0,1.0,1.0,2.0,217.0,3.08728
num_secondary_outcomes,100000.0,3.87528,0.0,0.0,2.0,5.0,349.0,6.41961
num_collaborators,100000.0,0.58787,0.0,0.0,0.0,1.0,69.0,1.570371
min_age,90808.0,20.214199,0.0,18.0,18.0,18.0,90.0,10.691286


**Observation**
1. Study timeline fields  
The date fields (`start_date`, `primary_completion_date`, `completion_date`) are observed to span several decades.
2. Outcome-related variables
    - The variables `num_primary_outcomes` and `num_secondary_outcomes` exhibit strong right-skewness.
    - Most trials report few outcomes, while a small number contain unusually large counts, likely corresponding to complex or multi-center study designs.
3. Age eligibility fields 
    - The fields `min_age` and `max_age` contain substantial missingness (missing counts: 9192 for min_age and 48618 for max_age). This pattern indicates that age restrictions are not consistently reported across studies. 
    - The broad range in max_age (1 to 150 years) suggests variability in inclusion criteria and potential data entry noise.
4. Location-based variables
    - Variables including `num_locations` and `num_countries` show highly skewed distributions.
    - While many trials operate in a single site or country, a small subset involves hundreds or even thousands of locations, reflecting large-scale global trials.
5. MeSH annotation variables
    - The variables `num_condition_mesh_terms` and `num_intervention_mesh_terms` show relatively narrow ranges, suggesting consistent annotation practices.
    - However, the presence of zero values suggests either missing MeSH mappings or conditions/interventions that were not classified.
6. Clinical phase representation
    - The variable phase_num has many missing values because numerous trials are labeled `"N/A"`, meaning they do not belong to any clinical phase (e.g., device or observational studies).
    - These missing values are expected and do not indicate data quality issues.

### Categorical Summary
Frequency tables are produced for categorical variables.  
This procedure allows rare categories, dominant labels, and unexpected text variations to be identified.

In [8]:
categorical_cols = df.select_dtypes(include=["object", "category"]).columns

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(10))


--- nct_id ---
nct_id
NCT02979535    1
NCT03280082    1
NCT04732182    1
NCT00425282    1
NCT02284282    1
NCT06515782    1
NCT05197582    1
NCT07109882    1
NCT06212882    1
NCT06123182    1
Name: count, dtype: int64

--- overall_status ---
overall_status
COMPLETED                  54763
UNKNOWN                    15114
RECRUITING                 11701
TERMINATED                  5755
NOT_YET_RECRUITING          4519
ACTIVE_NOT_RECRUITING       3862
WITHDRAWN                   2805
ENROLLING_BY_INVITATION      846
SUSPENDED                    265
WITHHELD                     175
Name: count, dtype: int64

--- study_type ---
study_type
INTERVENTIONAL     76458
OBSERVATIONAL      23172
EXPANDED_ACCESS      195
NaN                  175
Name: count, dtype: int64

--- phases ---
phases
NaN               62014
PHASE2            11096
PHASE1             8261
PHASE3             7365
PHASE4             6124
PHASE1, PHASE2     2843
PHASE2, PHASE3     1255
EARLY_PHASE1       1042
Name: count, d

**Observation**  
The categorical summary reveals several important characteristics:

- **Many categorical fields contain missing`"NaN"` values after cleaning, especially** in `allocation`, `intervention_model`, `masking`, `primary_purpose`, `responsible_party_type`, and `sex_eligibility`.  
  These values will be imputed with `"Unknown"` to maintain interpretability.

- **The `phases` variable is highly imbalanced.**  
  A large portion of studies are labeled `"NaN"`  
  This is expected because many study types (e.g., observational, device, behavioral, or expanded-access trials) do not belong to traditional clinical phases, so their phase field is legitimately empty.  
  
  True clinical phases (`PHASE1`, `PHASE2`, `PHASE3`, `PHASE4`) represent a smaller portion of the dataset, while combined-phase labels such as `"PHASE1, PHASE2"` and `"PHASE2, PHASE3"` are relatively rare.
- **The `study_type` field is dominated by interventional trials**  
  This distribution reflects the structure of ClinicalTrials.gov, where most registered studies involve therapeutic interventions.

- **`overall_status` shows strong concentration in `COMPLETED` and `UNKNOWN`**, with additional rare operational labels such as `TERMINATED`, `WITHDRAWN`, and `SUSPENDED`.  
  These tail categories may require grouping in downstream modeling.

- **The `conditions` field is a high-cardinality multi-label variable**  
  Each study lists one or more conditions (stored as Python lists).    
  Because each list may contain a different combination of conditions, the variable is:
  - extremely high-cardinality,  
  - sparsely repeated across trials, and  
  - intrinsically *multi-label* rather than categorical.  
  As such, it should not be treated as a standard categorical feature; proper multi-label encoding (e.g., tokenization, multi-hot encoding, MeSH grouping) is required in later stages.

- **`sex_eligibility` is overwhelmingly “ALL”**, with only limited representation of single-sex trials.  
  This imbalance may influence feature relevance during modeling.

Overall, the categorical distribution is characterized by strong imbalance, high-cardinality fields, and structured missingness-patterns that justify the chosen imputation strategy and highlight the need for careful encoding during downstream analysis.

### Missing Values
Missing values are quantified across all columns.  
This assessment enables the selection of appropriate imputation strategies and highlights data sparsity issues that may affect modeling performance.

In [9]:
df.isna().sum()

nct_id                             0
overall_status                     0
start_date                       949
completion_date                 3041
primary_completion_date         3915
study_type                       175
phases                         62014
allocation                     41639
intervention_model             24728
masking                        24518
primary_purpose                24823
conditions                         0
num_conditions                     0
num_arms                           0
num_interventions                  0
num_primary_outcomes               0
num_secondary_outcomes             0
responsible_party_type          7881
lead_sponsor_class               175
num_collaborators                  0
healthy_volunteers              2568
sex_eligibility                  258
min_age                         9192
max_age                        48618
num_locations                      0
num_countries                      0
num_condition_mesh_terms           0
n

A structured imputation strategy is applied:
- Most numeric features are imputed using the median, providing robustness to skewed distributions.
- `phase_num` is treated separately: missing values correspond to trials with no defined clinical phase (e.g., observational or device studies), and are therefore imputed with **0** to preserve semantic meaning.
- Safe categorical fields are imputed using `"Unknown"` to preserve interpretability.
- Boolean fields are assigned default logical values.
- Date fields are left with missing values, as these often carry meaningful information about trial status (e.g., ongoing or withdrawn).

This step ensures that the dataset does not contain gaps that would interfere with model training or statistical procedures.

In [10]:
df_filled = df.copy()

# Identify numeric columns except phase_num
numeric_cols = df_filled.select_dtypes(include="number").columns
numeric_cols = numeric_cols.drop("phase_num")

# Fill numeric missing values EXCEPT phase_num
df_filled = fill_missing_median(df_filled, columns=numeric_cols)

# Fill phase_num separately (preserve meaning: no phase = 0)
df_filled["phase_num"] = df_filled["phase_num"].fillna(0)

# Identify safe categorical columns
safe_cat_cols = [
    c for c in df_filled.columns
    if df_filled[c].dtype == object
    and not df_filled[c].apply(lambda x: isinstance(x, (list, dict))).any()
]

# Fill categorical
df_filled[safe_cat_cols] = df_filled[safe_cat_cols].fillna("Unknown")
df_filled["sex_eligibility"] = df_filled["sex_eligibility"].cat.add_categories(["Unknown"]).fillna("Unknown")

# Fill boolean
df_filled["healthy_volunteers"] = df_filled["healthy_volunteers"].fillna(False)

df_filled.isna().sum()

nct_id                             0
overall_status                     0
start_date                       949
completion_date                 3041
primary_completion_date         3915
study_type                         0
phases                         62014
allocation                         0
intervention_model                 0
masking                            0
primary_purpose                    0
conditions                         0
num_conditions                     0
num_arms                           0
num_interventions                  0
num_primary_outcomes               0
num_secondary_outcomes             0
responsible_party_type             0
lead_sponsor_class                 0
num_collaborators                  0
healthy_volunteers                 0
sex_eligibility                    0
min_age                            0
max_age                            0
num_locations                      0
num_countries                      0
num_condition_mesh_terms           0
n

**Observation**  
As a result of this imputation pipeline, all feature columns except date fields and the phases category become fully complete (0 missing values). Date fields and phases retain structured missingness that carries meaningful information and should not be imputed.

### Outliers
Potential outliers are examined using the Interquartile Range (IQR) method.  

Outlier filtering is applied only to `num_conditions`, `num_arms`, and `num_locations` because these variables frequently contain extreme values that do not represent typical clinical trials (e.g., studies listing dozens of conditions, many arms, or hundreds of locations).  
Removing these extremes improves dataset stability.  

Other numeric fields are not filtered because their large values are domain-appropriate (e.g., age ranges, number of outcomes) or represent encoded categorical concepts (`phase_num`).  
Thus, limiting outlier removal to these three columns avoids unnecessary loss of valid data.

In [11]:
cols_to_check = ["num_conditions", "num_arms", "num_locations"]

df_no_outliers = df_filled.copy()

for col in cols_to_check:
    df_no_outliers = remove_outliers_iqr(df_no_outliers, col)

df_no_outliers.shape

(55874, 30)

## Data Validation
After all cleaning steps, the final dataset is validated to ensure full structural and numerical integrity.  
The following checks are performed:
- Previewing the first rows to confirm schema consistency
- Checking for remaining missing values
- Ensuring numeric fields contain no missing values
- Confirming the absence of infinite or negative values
- Verifying that the numeric ranges remain reasonable after outlier filtering

These checks confirm that the dataset is consistent and ready for downstream EDA and modeling.

In [12]:
df_no_outliers.head()

,nct_id,overall_status,start_date,completion_date,primary_completion_date,study_type,phases,allocation,intervention_model,masking,...,healthy_volunteers,sex_eligibility,min_age,max_age,num_locations,num_countries,num_condition_mesh_terms,num_intervention_mesh_terms,has_results,phase_num
0,NCT02979535,COMPLETED,2016-11-16,2019-03-25,2019-03-25,INTERVENTIONAL,PHASE3,RANDOMIZED,PARALLEL,NONE,...,True,FEMALE,9.0,14.0,1,1,2,2,True,3.0
1,NCT02074735,COMPLETED,2014-04-01,2016-11-01,2016-10-01,INTERVENTIONAL,PHASE4,RANDOMIZED,PARALLEL,TRIPLE,...,False,ALL,18.0,75.0,1,1,1,2,True,4.0
2,NCT05242835,RECRUITING,2023-02-03,2027-03-01,2027-01-01,INTERVENTIONAL,NaN,RANDOMIZED,PARALLEL,QUADRUPLE,...,False,ALL,18.0,60.0,1,1,2,0,False,0.0
3,NCT07113535,NOT_YET_RECRUITING,2025-08-01,2027-12-31,2027-06-30,INTERVENTIONAL,NaN,RANDOMIZED,PARALLEL,SINGLE,...,False,ALL,18.0,65.0,1,1,2,0,False,0.0
4,NCT01684735,COMPLETED,2012-03-01,2014-12-01,2014-12-01,OBSERVATIONAL,NaN,Unknown,Unknown,Unknown,...,False,FEMALE,18.0,65.0,1,1,1,0,False,0.0


In [13]:
df_no_outliers.isna().sum()

nct_id                             0
overall_status                     0
start_date                       314
completion_date                 1397
primary_completion_date         2021
study_type                         0
phases                         38633
allocation                         0
intervention_model                 0
masking                            0
primary_purpose                    0
conditions                         0
num_conditions                     0
num_arms                           0
num_interventions                  0
num_primary_outcomes               0
num_secondary_outcomes             0
responsible_party_type             0
lead_sponsor_class                 0
num_collaborators                  0
healthy_volunteers                 0
sex_eligibility                    0
min_age                            0
max_age                            0
num_locations                      0
num_countries                      0
num_condition_mesh_terms           0
n

In [14]:
validate_values(df_no_outliers)

{'num_numeric_columns': 13,
 'nan_count': 0,
 'inf_count': 0,
 'negative_count': 0,
 'min_value': 0.0,
 'max_value': 169.0}

### Export Cleaned Data
The fully cleaned and validated dataset is exported to the `processed` directory.  
This file will be used in subsequent notebooks for exploratory data analysis and predictive modeling.

In [15]:
OUTPUT_PATH = "../data/processed/clinical_trials_cleaned.csv"
df_no_outliers.to_csv(OUTPUT_PATH, index=False)
print("Saved to:", OUTPUT_PATH)

Saved to: ../data/processed/clinical_trials_cleaned.csv
